In [48]:
!pip install --upgrade pip
!pip install langchain-community langchain-text-splitters
!pip install faiss-cpu sentence-transformers transformers

In [62]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)


In [63]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline


In [64]:
# load the data source
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Multi-agent_system",)
)
docs = loader.load()
print(f"Loaded document with {len(docs[0].page_content)} characters")

Loaded document with 27898 characters


In [65]:
#let's spit in chunks sizes
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
print(f"Split document into {len(all_splits)} sub-documents.")

Split document into 39 sub-documents.


In [66]:
#Creation of embeddings using HuggingFace
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [67]:
#creation of the vector store
vector_store = FAISS.from_documents(
    documents=all_splits,
    embedding=embeddings
)
print(f"Vector store created with {len(all_splits)} documents.")

Vector store created with 39 documents.


In [69]:
#let's create the retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [70]:
class QuestionAgent:
    def ask(self):

        query = (
            "What  is  a multi−agent  system, and how is it related  "
            "to  large  language models  (LLMs) ?"
        )
        return query


In [90]:
class AnswerAgent:
    def __init__(self):
        # HuggingFace text generation model
        self.pipe = pipeline(
            "text2text-generation",
            model="google/flan-t5-base",
            max_length=512
        )

    def answer(self, question):
        print("\nANSWER AGENT:")
        print("(Processing with retrieval steps...)")

        docs = retriever.invoke(question)
        print(f"Retrieved {len(docs)} documents from the vector store.")

        for i, doc in enumerate(docs):
            snippet = doc.page_content[:200].replace("\n", " ")
            print(f"Document {i+1} snippet: {snippet}...")

        context = "\n".join([doc.page_content[:1000] for doc in docs])



        prompt = f"""
        Question: {question}

        Context:
        {context}

        Answer:
        """

        output = self.pipe(prompt)[0]["generated_text"]

        print("\nFinal Answer :")
        return output
        print(output + " [Comprehensive answer based on retrieved Wikipedia content]")



In [91]:

print("=" * 70)
print("Two-Agent Conversation :  Question Asker <-> Answer Provider")
print("=" * 70)

question_agent = QuestionAgent()
answer_agent = AnswerAgent()

#let's print the output using the teacher's format
# Question Agent output
question = question_agent.ask()
print("\nQUESTION AGENT:")
print(f"Question : {question}")


# Answer Agent output
response = answer_agent.answer(question)
print(response)

print("=" * 70)
print("Conversation Complete !")
print("=" * 70)

Two-Agent Conversation :  Question Asker <-> Answer Provider

QUESTION AGENT:
Question : What  is  a multi−agent  system, and how is it related  to  large  language models  (LLMs) ?

ANSWER AGENT:
(Processing with retrieval steps...)
Retrieved 4 documents from the vector store.
Document 1 snippet: vte Simple reflex agent Learning agent A multi-agent system (MAS or "self-organized system") is a computerized system composed of multiple interacting intelligent agents.[1][2] Multi-agent systems can...
Document 2 snippet: ^ "IEEE IES Technical Committee on Industrial Agents (TC-IA)". tcia.ieee-ies.org. Retrieved February 19, 2019.  ^ "CAMEL: Finding the Scaling Law of Agents. The first and the best multi-agent framewor...
Document 3 snippet: Further reading[edit] Wooldridge, Michael (2002). An Introduction to MultiAgent Systems. John Wiley & Sons. p. 366. ISBN 978-0-471-49691-5. Shoham, Yoav; Leyton-Brown, Kevin (2008). Multiagent Systems...
Document 4 snippet: ^ Wooldridge, Michael (2002).